#### 指数、国债数据获取

In [2]:
import tushare as ts
import pandas as pd
import akshare as ak
import datetime
from config.config import *

In [2]:
BENCHMARK_COL = "benchmark"   # 例如 "业绩比较基准" / "基准" 等
index_map = {
    "沪深300": "000300.SH",
    "恒生指数": "HSI.HI",
    "中证港股通": "930930.CSI",
    "中证全债": "H11001.CSI",
    "中证综合债": "H11009.CSI",
    "上证国债": "000012.SH",
    "中证内地消费主题": "000932.SH",

    "中债新综合财富（总值）": "CBA00101.CS",
    
    "中债新综合全价（总值）": "CBA00103.CS",

    "中债综合财富(总值)": "CBA00201.CS",
    "中债综合指数(总财富)": "CBA00201.CS",
    "中债综合指数收益率": "CBA00201.CS",

    "中债综合指数(全价)": "CBA00203.CS",
    "中债综合(全价)": "CBA00203.CS",
    "中债综合全价": "CBA00203.CS",
    "中债-综合全价（总值）": "CBA00203.CS",
    "中国债券综合全价": "CBA00203.CS",
    "中债总指数(全价)": "CBA00203.CS",
    "中债-总全价(总值)": "CBA00203.CS",

    "中债综合财富(1年以下)中债": "CBA00211.CS",
    "中债综合财富(1-3年)": "CBA00221.CS",

}

In [3]:
INDEX_CODES = {
    # "上证国债": "000012.SH",
    "沪深300": "000300.SH",
    "中证内地消费主题": "000932.SH",
    "中证港股通": "930930.CSI",
    "中证全债": "H11001.CSI",
    "中证综合债": "H11009.CSI",
    "上证国债": "000012.SH"
}



# 时间范围（可根据需求调整，建议覆盖基金成立日至今）
# START_DATE = "20200101"  # 起始日期
# END_DATE = datetime.today().date()    # 结束日期（当前日期）

# -------------------------- 2. 批量获取指数数据并计算累计收益率 --------------------------
def get_index_cum_return(index_code):
    """
    从Tushare获取指数数据，计算累计收益率
    返回：Series（索引为日期，值为累计收益率）
    """
    # 获取指数日线数据（adj_close为复权收盘价，用于计算收益率）
    df = pro.index_daily(
        ts_code=index_code,
        # start_date=start_date,
        # end_date=end_date,
        fields="trade_date, close"  # 仅需日期和复权收盘价
    )
    
    # 数据预处理：日期转datetime格式，按日期升序排序
    df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
    df = df.sort_values("trade_date").set_index("trade_date")
    
    return df

# 获取所有指数的累计收益率数据
for name, code in INDEX_CODES.items():
    cum_return_series = get_index_cum_return(code)
    cum_return_series.to_csv(f"index/{code}.csv")

In [4]:
# start_date = '20250101'
# end_date = '20251118'
def fx_mid(ts_code):
    """从 fx_daily 取 bid/ask，返回按日期排序的中间价序列：['trade_date','mid']"""
    fx = pro.fx_daily(
        ts_code=ts_code,
        fields="trade_date,bid_close,ask_close"
    )
    fx["trade_date"] = pd.to_datetime(fx["trade_date"])
    fx["mid"] = (fx["bid_close"] + fx["ask_close"]) / 2.0
    fx = fx.sort_values("trade_date")[["trade_date", "mid"]].reset_index(drop=True)
    return fx

def get_hsi_cny(hsi_code="HSI"):  # 如报错，可尝试 'HSI.GI' 等
    # 1) 恒生指数(HKD)
    hsi = pro.index_global(
        ts_code=hsi_code,
        fields="trade_date,close"
    )
    hsi["trade_date"] = pd.to_datetime(hsi["trade_date"])
    hsi = hsi.sort_values("trade_date")

    # 2) 构造 HKD/CNY ~ USDCNH / USDHKD （若有 USDCNY.FX，更换为 USDCNY）
    usdcnh = fx_mid("USDCNH.FXCM")  # CNY(离岸) / USD
    usdhkd = fx_mid("USDHKD.FXCM")  # HKD / USD

    fx = pd.merge(usdcnh, usdhkd, on="trade_date", how="inner", suffixes=("_usdcnh", "_usdhkd"))
    fx["HKD_CNY"] = fx["mid_usdcnh"] / fx["mid_usdhkd"]  # (CNY/USD) / (HKD/USD) = CNY/HKD
    # 注意：这里得到的是 CNY/HKD，如果你要“1 HKD = ? CNY”，需要取其倒数
    fx["HKD_to_CNY"] = 1.0 / fx["HKD_CNY"]  # 即 HKD/CNY

    # 3) 生成人民币口径恒指：HSI(HKD) × HKD/CNY
    df = pd.merge(hsi[["trade_date","close"]], fx[["trade_date","HKD_to_CNY"]], on="trade_date", how="inner")
    df["HSI_CNY"] = df["close"] * df["HKD_to_CNY"]
    df['close'] = df['HSI_CNY']
    return df[["trade_date", "close"]].reset_index(drop=True)

hsi_df = get_hsi_cny()
hsi_df.to_csv("index/HSI.HI.csv", index=False)

In [8]:
df_wealth = ak.bond_composite_index_cbond(
        indicator="财富",   # 财富指数
        period="总值"       # 总值
    )
df_wealth

/opt/anaconda3/lib/python3.13/site-packages/akshare/bond/bond_cbond.py:160: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  temp_df["date"] = pd.to_datetime(temp_df["date"], unit="ms").dt.date


,date,value
0,2002-01-03,99.9731
1,2002-01-06,100.0149
2,2002-01-07,99.8273
3,2002-01-08,100.0203
4,2002-01-09,99.9317
...,...,...
6056,2026-03-22,256.8873
6057,2026-03-23,256.9822
6058,2026-03-24,257.0406
6059,2026-03-25,257.1216


In [9]:
def save_clean(df: pd.DataFrame,code: str):
    print(df.head())
    df["trade_date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    df = df.sort_values("trade_date").reset_index(drop=True)
    df = df[["trade_date", "value"]]
    df = df.set_index("trade_date")
    df.to_csv(f"index/{code}.csv", index=True)
    
def zhongzhai():
    # 中债-综合指数：财富指数（总值）
    df_wealth = ak.bond_composite_index_cbond(
        indicator="财富",   # 财富指数
        period="总值"       # 总值
    )
    print("财富指数-总值：")
    # print(df_wealth.tail())
    df_wealth.to_csv('CBA00201.CS.csv')
    save_clean(df_wealth, "CBA00201.CS.csv")

    # 中债-综合指数：全价指数（总值）
    df_full = ak.bond_composite_index_cbond(
        indicator="全价",   # 全价指数
        period="总值"
    )
    print("全价指数-总值：")
    # print(df_full.tail())
    df_full.to_csv('CBA00203.CS.csv')
    save_clean(df_full, "CBA00203.CS.csv")

    # 中债-综合指数：净价指数（总值）——顺带一起写了
    df_net = ak.bond_composite_index_cbond(
        indicator="净价",   # 净价指数
        period="总值"
    )
    print("净价指数-总值：")
    # print(df_net.tail())
    save_clean(df_net, "CBA00202.CS.csv")

    df_1 = ak.bond_composite_index_cbond(
        indicator="财富",
        period="1年以下"
    )
    save_clean(df_1, "CBA00211.CS.csv")

    df_13 = ak.bond_composite_index_cbond(
        indicator="财富",
        period="1-3年"
    )
    save_clean(df_13, "CBA00221.CS.csv")
    

    df_new_wealth = ak.bond_new_composite_index_cbond(
        indicator="财富",
        period="总值"
    )
    print(df_new_wealth.tail())
    save_clean(df_new_wealth, "CBA00101.CS.csv")


    df_new_full = ak.bond_new_composite_index_cbond(
        indicator="全价",
        period="总值"
    )
    print(df_new_full.tail())
    save_clean(df_new_full, "CBA00103.CS.csv")

zhongzhai()

/opt/anaconda3/lib/python3.13/site-packages/akshare/bond/bond_cbond.py:160: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  temp_df["date"] = pd.to_datetime(temp_df["date"], unit="ms").dt.date


财富指数-总值：
         date     value
0  2002-01-03   99.9731
1  2002-01-06  100.0149
2  2002-01-07   99.8273
3  2002-01-08  100.0203
4  2002-01-09   99.9317


/opt/anaconda3/lib/python3.13/site-packages/akshare/bond/bond_cbond.py:160: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  temp_df["date"] = pd.to_datetime(temp_df["date"], unit="ms").dt.date


全价指数-总值：
         date     value
0  2002-01-03   99.9731
1  2002-01-06  100.0149
2  2002-01-07   99.8273
3  2002-01-08  100.0203
4  2002-01-09   99.9317


/opt/anaconda3/lib/python3.13/site-packages/akshare/bond/bond_cbond.py:160: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  temp_df["date"] = pd.to_datetime(temp_df["date"], unit="ms").dt.date


净价指数-总值：
         date    value
0  2002-01-03  99.9350
1  2002-01-06  99.9491
2  2002-01-07  99.7497
3  2002-01-08  99.9358
4  2002-01-09  99.8367


/opt/anaconda3/lib/python3.13/site-packages/akshare/bond/bond_cbond.py:160: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  temp_df["date"] = pd.to_datetime(temp_df["date"], unit="ms").dt.date


         date     value
0  2002-01-03   99.9642
1  2002-01-06   99.9349
2  2002-01-07  100.2792
3  2002-01-08  100.4043
4  2002-01-09  100.4195


/opt/anaconda3/lib/python3.13/site-packages/akshare/bond/bond_cbond.py:160: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  temp_df["date"] = pd.to_datetime(temp_df["date"], unit="ms").dt.date


         date     value
0  2002-01-03   99.8434
1  2002-01-06   99.8191
2  2002-01-07  100.3705
3  2002-01-08  100.5468
4  2002-01-09  100.5791
            date     value
6056  2026-03-23  250.5674
6057  2026-03-24  250.6598
6058  2026-03-25  250.7166
6059  2026-03-26  250.7956
6060  2026-03-27  250.8172
         date     value
0  2002-01-04   99.9731
1  2002-01-07  100.0149
2  2002-01-08   99.8273
3  2002-01-09  100.0203
4  2002-01-10   99.9317
            date     value
6056  2026-03-23  128.9628
6057  2026-03-24  128.9991
6058  2026-03-25  128.9993
6059  2026-03-26  129.0286
6060  2026-03-27  129.0318
         date     value
0  2002-01-04   99.9731
1  2002-01-07  100.0149
2  2002-01-08   99.8273
3  2002-01-09  100.0203
4  2002-01-10   99.9317


In [6]:
# 一年期国债 Y010101.CB
gz1 = pro.yc_cb(ts_code='1001.CB', curve_term = '1.0', offset = '1500',curve_type = '0')
gz1

,trade_date,ts_code,curve_name,curve_type,curve_term,yield
0,20200325,1001.CB,中债国债收益率曲线,0,1.0,1.7686
1,20200324,1001.CB,中债国债收益率曲线,0,1.0,1.8081
2,20200323,1001.CB,中债国债收益率曲线,0,1.0,1.8482
3,20200320,1001.CB,中债国债收益率曲线,0,1.0,1.8682
4,20200319,1001.CB,中债国债收益率曲线,0,1.0,1.8853
...,...,...,...,...,...,...
945,20160617,1001.CB,中债国债收益率曲线,0,1.0,2.3989
946,20160616,1001.CB,中债国债收益率曲线,0,1.0,2.4006
947,20160615,1001.CB,中债国债收益率曲线,0,1.0,2.4071
948,20160614,1001.CB,中债国债收益率曲线,0,1.0,2.4087


In [7]:
gz0 = pd.read_csv("index/GuoY_1.csv")
gz0

,trade_date,ts_code,curve_name,curve_type,curve_term,yield
0,20260327,1001.CB,中债国债收益率曲线,0,1.0,1.2518
1,20260326,1001.CB,中债国债收益率曲线,0,1.0,1.2568
2,20260325,1001.CB,中债国债收益率曲线,0,1.0,1.2668
3,20260324,1001.CB,中债国债收益率曲线,0,1.0,1.2618
4,20260323,1001.CB,中债国债收益率曲线,0,1.0,1.2568
...,...,...,...,...,...,...
1995,20180330,1001.CB,中债国债收益率曲线,0,1.0,3.3221
1996,20180329,1001.CB,中债国债收益率曲线,0,1.0,3.3221
1997,20180328,1001.CB,中债国债收益率曲线,0,1.0,3.3221
1998,20180327,1001.CB,中债国债收益率曲线,0,1.0,3.3221


In [11]:
gz2 = pd.concat([gz0, gz1], ignore_index=True)
gz2['trade_date'] = pd.to_datetime(gz2['trade_date'], format="%Y%m%d")
gz_final = gz2.drop_duplicates(subset=["trade_date"], keep="last")
gz_final

,trade_date,ts_code,curve_name,curve_type,curve_term,yield
0,2026-03-27,1001.CB,中债国债收益率曲线,0,1.0,1.2518
1,2026-03-26,1001.CB,中债国债收益率曲线,0,1.0,1.2568
2,2026-03-25,1001.CB,中债国债收益率曲线,0,1.0,1.2668
3,2026-03-24,1001.CB,中债国债收益率曲线,0,1.0,1.2618
4,2026-03-23,1001.CB,中债国债收益率曲线,0,1.0,1.2568
...,...,...,...,...,...,...
2945,2016-06-17,1001.CB,中债国债收益率曲线,0,1.0,2.3989
2946,2016-06-16,1001.CB,中债国债收益率曲线,0,1.0,2.4006
2947,2016-06-15,1001.CB,中债国债收益率曲线,0,1.0,2.4071
2948,2016-06-14,1001.CB,中债国债收益率曲线,0,1.0,2.4087


出问题了，一年期国债最早只有2016-06-13

In [12]:
gz_final.to_csv('index/GuoY_1.csv',index=False)

In [4]:
import pandas as pd

# 从官网爬到的，https://www.pbc.gov.cn/zhengcehuobisi/125207/125213/125440/125838/125888/2968982/index.html
data = {
    "trade_date": [
        "1990.04.15", "1990.08.21", "1991.04.21", "1993.05.15", "1993.07.11",
        "1996.05.01", "1996.08.23", "1997.10.23", "1998.03.25", "1998.07.01",
        "1998.12.07", "1999.06.10", "2002.02.21", "2004.10.29", "2006.08.19",
        "2007.03.18", "2007.05.19", "2007.07.21", "2007.08.22", "2007.09.15",
        "2007.12.21", "2008.10.09", "2008.10.30", "2008.11.27", "2008.12.23",
        "2010.10.20", "2010.12.26", "2011.02.09", "2011.04.06", "2011.07.07",
        "2012.06.08", "2012.07.06", "2014.11.22", "2015.03.01", "2015.05.11",
        "2015.06.28", "2015.08.26", "2015.10.24"
    ],
    "yield": [
        6.30, 4.32, 3.24, 4.86, 6.66,
        4.86, 3.33, 2.88, 2.88, 2.79,
        2.79, 1.98, 1.71, 1.71, 1.80,
        1.98, 2.07, 2.34, 2.61, 2.88,
        3.33, 3.15, 2.88, 1.98, 1.71,
        1.91, 2.25, 2.60, 2.85, 3.10,
        2.85, 2.60, 2.35, 2.10, 1.85,
        1.60, 1.35, 1.10
    ]
}

df = pd.DataFrame(data)
df['yield'] = df['yield'] / 100.0  # 转换为小数形式
df["trade_date"] = pd.to_datetime(df["trade_date"])
df = df.sort_values("trade_date").reset_index(drop=True)  # 按时间排序
start_date = df["trade_date"].min()
end_date = pd.to_datetime("2026-03-27")
full_dates = pd.date_range(start=start_date, end=end_date, freq="D")
df_full = pd.DataFrame({"trade_date": full_dates})

# 合并并前向填充（ffill）
df_daily = pd.merge(df_full, df, on="trade_date", how="left")
df_daily["yield"] = df_daily["yield"].ffill()  # 用前一个有效值填充

df_daily

,trade_date,yield
0,1990-04-15,0.063
1,1990-04-16,0.063
2,1990-04-17,0.063
3,1990-04-18,0.063
4,1990-04-19,0.063
...,...,...
13126,2026-03-23,0.011
13127,2026-03-24,0.011
13128,2026-03-25,0.011
13129,2026-03-26,0.011


In [5]:
df_daily.to_csv("index/3m_deposit_rate.csv", index=False)